# Fake News Detection Model Training (Google Colab)
### Classical NLP: TF-IDF (100k features) + Balanced Logistic Regression (C=2.0)
**Dataset**: WELFake Dataset (~72,134 news articles: 0 = FAKE, 1 = REAL)

This notebook trains the model remotely in Google Colab and exports the production artifacts for your local Streamlit application:
- `fake_news_model.joblib`
- `tfidf_vectorizer.joblib`
- `evaluation.json`
- `confusion_matrix.png`

## 1. Setup Environment & Install Dependencies

In [ ]:
!pip install --upgrade scikit-learn pandas numpy joblib matplotlib

import os
from pathlib import Path
os.makedirs("/content/models", exist_ok=True)
os.makedirs("/content/data", exist_ok=True)
print("Environment initialized!")

## 2. Provide Dataset
Upload your `WELFake_Dataset.csv` using one of the options below:
- **Option A**: Click the folder icon on the left panel in Colab and drag & drop `WELFake_Dataset.csv` into `/content/data/`
- **Option B**: Run the upload cell below to browse and select the file from your computer.

In [ ]:
# Run this cell to upload WELFake_Dataset.csv directly from your computer:
from google.colab import files
import os

# Check if dataset already exists in /content/data or /content
dataset_found = False
for p in ["/content/data/WELFake_Dataset.csv", "/content/WELFake_Dataset.csv"]:
    if os.path.exists(p):
        print(f"Found dataset at: {p}")
        dataset_found = True
        break

if not dataset_found:
    print("Please select WELFake_Dataset.csv to upload:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        os.rename(fn, f"/content/data/{fn}")
        print(f"Saved to /content/data/{fn}")

## 3. Train Model (Preprocessing, TF-IDF, Logistic Regression & Evaluation)

In [ ]:
import html
import json
import logging
from pathlib import Path
import re
import time
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
from sklearn.model_selection import train_test_split

# Preprocessing regex
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
HTML_TAG_PATTERN = re.compile(r"<.*?>", re.DOTALL)
SPECIAL_CHAR_PATTERN = re.compile(r"[^a-zA-Z0-9\s]")
WHITESPACE_PATTERN = re.compile(r"\s+")

def clean_text(text):
    if text is None or not isinstance(text, str) or not text.strip():
        return ""
    cleaned = html.unescape(text)
    cleaned = HTML_TAG_PATTERN.sub(" ", cleaned)
    cleaned = URL_PATTERN.sub(" ", cleaned)
    cleaned = cleaned.lower()
    cleaned = SPECIAL_CHAR_PATTERN.sub(" ", cleaned)
    return WHITESPACE_PATTERN.sub(" ", cleaned).strip()

# 1. Locate dataset CSV
data_path = None
for candidate in Path("/content").glob("**/*.csv"):
    try:
        cols = pd.read_csv(candidate, nrows=2).columns.str.strip().str.lower().tolist()
        if "title" in cols and "text" in cols and "label" in cols:
            data_path = candidate
            break
    except Exception:
        continue

assert data_path is not None, "Could not find a CSV dataset with columns ['title', 'text', 'label']."
print(f"Using dataset: {data_path}")

# 2. Load & Clean
print("Loading dataset...")
df = pd.read_csv(data_path)
df = df.rename(columns={c: c.strip().lower() for c in df.columns})
df["title"] = df["title"].fillna("").astype(str)
df["text"] = df["text"].fillna("").astype(str)
df = df[df["label"].isin([0, 1])].copy()
df["label"] = df["label"].astype(int)

print("Preprocessing text...")
df["combined_text"] = [
    f"{clean_text(t)} {clean_text(b)}".strip()
    for t, b in zip(df["title"], df["text"])
]
df = df[df["combined_text"].str.strip() != ""].copy()
before_dedup = len(df)
df = df.drop_duplicates(subset=["combined_text"]).copy()
print(f"Total cleaned records: {len(df)} (Dropped {before_dedup - len(df)} duplicates)")
print(f"FAKE (0): {(df['label'] == 0).sum()} | REAL (1): {(df['label'] == 1).sum()}")

# 3. Stratified 80/10/10 Split
X = df["combined_text"]
y = df["label"]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print(f"Split: {len(X_train)} Train | {len(X_val)} Val | {len(X_test)} Test")

# 4. Fit TF-IDF on Train Only
print("Fitting TF-IDF Vectorizer (100,000 features, unigrams + bigrams)...")
vectorizer = TfidfVectorizer(max_features=100000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(X_train)
print(f"TF-IDF Vocabulary: {len(vectorizer.vocabulary_)} terms")

# 5. Train Logistic Regression
print("Training Logistic Regression classifier (balanced, C=2.0)...")
clf = LogisticRegression(C=2.0, class_weight="balanced", max_iter=1000, random_state=42, n_jobs=-1)
clf.fit(X_train_vec, y_train)

# 6. Evaluate on Test Split
print("Evaluating on test set...")
X_test_vec = vectorizer.transform(X_test)
y_pred = clf.predict(X_test_vec)

acc = float(accuracy_score(y_test, y_pred))
macro_f1 = float(f1_score(y_test, y_pred, average="macro"))
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = [int(v) for v in cm.ravel()]

print("=" * 60)
print(f"TEST SET ACCURACY: {acc * 100:.2f}%")
print(f"MACRO F1-SCORE:   {macro_f1:.4f}")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=["FAKE (0)", "REAL (1)"], digits=4))

# 7. Save Production Artifacts
out_dir = Path("/content/models")
joblib.dump(vectorizer, out_dir / "tfidf_vectorizer.joblib", compress=3)
joblib.dump(clf, out_dir / "fake_news_model.joblib", compress=3)

eval_data = {
    "model_name": "TF-IDF + Logistic Regression",
    "dataset": "WELFake Benchmark Dataset",
    "evaluated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "test_samples": int(len(y_test)),
    "metrics": {
        "accuracy": round(acc, 4),
        "macro_f1": round(macro_f1, 4),
        "real_class": {"precision": round(float(precision_score(y_test, y_pred, pos_label=1)), 4), "recall": round(float(recall_score(y_test, y_pred, pos_label=1)), 4), "f1_score": round(float(f1_score(y_test, y_pred, pos_label=1)), 4), "support": int((y_test == 1).sum())},
        "fake_class": {"precision": round(float(precision_score(y_test, y_pred, pos_label=0)), 4), "recall": round(float(recall_score(y_test, y_pred, pos_label=0)), 4), "f1_score": round(float(f1_score(y_test, y_pred, pos_label=0)), 4), "support": int((y_test == 0).sum())}
    },
    "confusion_matrix": {"true_negatives": tn, "false_positives": fp, "false_negatives": fn, "true_positives": tp, "matrix": [[tn, fp], [fn, tp]], "labels": ["FAKE", "REAL"]}
}
with open(out_dir / "evaluation.json", "w") as f:
    json.dump(eval_data, f, indent=2)

# Plot Confusion Matrix
fig, ax = plt.subplots(figsize=(5, 4))
cax = ax.matshow(cm, cmap=plt.cm.Blues, alpha=0.8)
fig.colorbar(cax)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=13, weight="bold")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["FAKE", "REAL"])
ax.set_yticklabels(["FAKE", "REAL"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.title("WELFake Test Confusion Matrix", pad=15)
plt.tight_layout()
plt.savefig(out_dir / "confusion_matrix.png", dpi=200)
plt.show()

print("\nSUCCESS: All artifacts written to /content/models/")

## 4. Download Trained Artifacts to Your Computer
Run the cell below to download the trained model and evaluation files directly to your Downloads folder.
Then, move them into your local `Fake-News-Detection-System/models/` folder.

In [ ]:
from google.colab import files

print("Downloading fake_news_model.joblib...")
files.download("/content/models/fake_news_model.joblib")

print("Downloading tfidf_vectorizer.joblib...")
files.download("/content/models/tfidf_vectorizer.joblib")

print("Downloading evaluation.json...")
files.download("/content/models/evaluation.json")

print("Downloading confusion_matrix.png...")
files.download("/content/models/confusion_matrix.png")

print("Done! Drop these files into your local 'models/' directory.")